# Generate a bird with the trained Stage-I StackGAN

This notebook loads the EMA weights from `runs/stage1/best_val.pt`, asks for a caption, finds its exact CUB text embedding, and generates a 64×64 image.

> **Important:** this repository has precomputed char-CNN/RNN embeddings, but not the pretrained text encoder. Therefore the prompt must be one of the captions in `birds/text_c10`. Arbitrary new text cannot be encoded correctly by this model. Try the first line from `birds/example_captions.txt`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'models.py').is_file():
    raise RuntimeError('Start Jupyter from the project root containing models.py.')
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display

from models import Stage1_Generator

CHECKPOINT_PATH = PROJECT_ROOT / 'runs' / 'stage1' / 'best_val.pt'
TEXT_ROOT = PROJECT_ROOT / 'birds' / 'text_c10'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42  # Change this to make a different image for the same caption.

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT_PATH}')
print(f'Using {DEVICE} and checkpoint: {CHECKPOINT_PATH}')


In [ ]:
def normalize_caption(text: str) -> str:
    return ' '.join(text.casefold().split())

def build_caption_index():
    """Map normalized CUB captions to (split, image index, caption index, original text)."""
    image_locations = {}
    for split in ('train', 'test'):
        filenames = pd.read_pickle(PROJECT_ROOT / 'birds' / split / 'filenames.pickle')
        for image_index, filename in enumerate(filenames):
            image_locations[filename.replace('\\', '/')] = (split, image_index)

    index = {}
    for text_path in TEXT_ROOT.rglob('*.txt'):
        filename = text_path.relative_to(TEXT_ROOT).with_suffix('').as_posix()
        if filename not in image_locations:
            continue
        split, image_index = image_locations[filename]
        captions = text_path.read_text(encoding='utf-8').splitlines()
        for caption_index, caption in enumerate(captions[:10]):
            index.setdefault(normalize_caption(caption), (split, image_index, caption_index, caption))
    return index

CAPTION_INDEX = build_caption_index()
print(f'Indexed {len(CAPTION_INDEX):,} unique captions.')


In [ ]:
# Enter an exact caption from birds/text_c10. The default is a verified example.
DEFAULT_PROMPT = 'this bird has a bright yellow body, with brown on its crown and wings.'
PROMPT = input(f'Caption [{DEFAULT_PROMPT}]: ').strip() or DEFAULT_PROMPT
prompt_key = normalize_caption(PROMPT)

if prompt_key not in CAPTION_INDEX:
    suggestions = [original for key, (*_, original) in CAPTION_INDEX.items() if prompt_key in key][:5]
    message = 'No exact precomputed embedding exists for this prompt.'
    if suggestions:
        message += '\nClosest substring matches:\n- ' + '\n- '.join(suggestions)
    raise ValueError(message)

split, image_index, caption_index, matched_caption = CAPTION_INDEX[prompt_key]
print(f'Matched {split} image #{image_index}, caption #{caption_index}: {matched_caption}')


In [ ]:
# Load only the embedding file that contains the selected caption.
embeddings = np.asarray(
    pd.read_pickle(PROJECT_ROOT / 'birds' / split / 'char-CNN-RNN-embeddings.pickle'),
    dtype=np.float32,
)
embedding = torch.from_numpy(embeddings[image_index, caption_index]).unsqueeze(0).to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
training_args = checkpoint.get('args', {})
model = Stage1_Generator(
    noise_size=training_args.get('noise_size', 100),
    base_channels=training_args.get('base_channels', 64),
).to(DEVICE)
# EMA is preferred; fall back for older compatible checkpoints.
model.load_state_dict(checkpoint.get('generator_ema', checkpoint['generator']))
model.eval()
print(f'Loaded epoch {checkpoint.get("epoch", "unknown")} EMA generator.')


In [ ]:
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

with torch.no_grad():
    image, _, _ = model(embedding)

# Convert the model's [-1, 1] output to a standard RGB PNG.
image_uint8 = ((image[0].detach().cpu().clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8)
pil_image = Image.fromarray(image_uint8.permute(1, 2, 0).numpy())
output_path = PROJECT_ROOT / 'generated_stage1.png'
pil_image.save(output_path)
display(pil_image.resize((512, 512)))
print(f'Saved: {output_path}')
